# Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import torch
from hydra.utils import instantiate
from omegaconf import OmegaConf

from src.utils.notebook_setup import init_nlp_notebook


# 1. Вычисляем корень проекта ОДИН РАЗ
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 2. Регистрируем все резолверы ГЛОБАЛЬНО
# Теперь ${paths.data_dir} будет подтягиваться из конфига,
# а если Hydra падает, мы подставляем ${PROJECT_ROOT}
OmegaConf.register_new_resolver("project_root", lambda: str(PROJECT_ROOT))
OmegaConf.register_new_resolver("hydra", lambda path: str(PROJECT_ROOT), replace=True)
OmegaConf.register_new_resolver("now", lambda *args: "now", replace=True)

# 3. Инициализируем Hydra
cfg = init_nlp_notebook()

# 4. Трюк: принудительно подменяем переменную paths, если она не разрешилась
# Это поможет конфигу увидеть, где лежат данные, без правок yaml-файлов
if "paths" not in cfg:
    cfg.paths = OmegaConf.create()
cfg.paths.data_dir = str(PROJECT_ROOT / "data")

device = "cuda" if torch.cuda.is_available() else "cpu"

NLP Environment ready. Root: c:\fake-news-detection-ml-system


# Data init and collator

In [ ]:
from src.core.data.builder import NLPDataModule  # noqa: E402


tokenizer = instantiate(cfg.model.tokenizer).build()

datamodule = NLPDataModule(
    data_cfg=cfg.data,
    tokenizer=tokenizer
)

datamodule.prepare_data()
datamodule.setup(stage="validate")
val_dataloader = datamodule.val_dataloader()
sample_batch = next(iter(val_dataloader))

print("Keys in batch:", sample_batch.keys())
print("Input IDs shape:", sample_batch["input_ids"].shape)
if "labels" in sample_batch:
    print("Labels shape:", sample_batch["labels"].shape)

INFO:src.core.models.tokenization:Загрузка токенизатора: DeepPavlov/rubert-base-cased
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/DeepPavlov/rubert-base-cased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/DeepPavlov/

Keys in batch: KeysView({'input_ids': tensor([[   101,  12032,  14298,  11270,  19085,  12111,  12639,   7491,  10623,
          11196,    248,  10622,  11324,  51965,   7491,  11664,  11796,  11715,
            257,  10618,  10617,  13447,  64824,   7729,    248,  10646,  10710,
          28556,  10228,  14958,  10966,  11869,   7159,  10626,  13447,  64824,
          10617,    262,  17476,  11220,  10617,    246,    239,  13685,  13089,
          13075,  11353,  10617,  16010,  26894,  59887,  11796,  17533,  11423,
          10626,  39941,   6722,  10617,  21018,  10414,  10783,  10676,  34476,
          73406,  20925,    269,  10647,  10617,    254,    237,    239,  10681,
          18026,  11690,  14107,   6931,   7729,  12362,  12186,  10708,  10937,
          13220,  37132,   3362,  10708,  10937,  13220,  37132,   3362,  10647,
          10783,  10783,  10636,  13332,  21996,  19148,  10681,    118,  25612,
            268,  33485,   8937,    118,  12889,  10880,   7729,  10647

# Model init

In [4]:
# Правильная проверка forward pass через NLPModel
base_model = instantiate(cfg.model.builder, tokenizer=tokenizer).build()
nlp_model = instantiate(cfg.model_module, model=base_model)
nlp_model.to(device)
nlp_model.eval()

INFO:src.core.models.builder:Загрузка модели из: DeepPavlov/rubert-base-cased
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18081.25it/s]
[transforme

trainable params: 296,450 || all params: 178,151,428 || trainable%: 0.1664


NLPModel(
  (model): PeftModelForSequenceClassification(
    (base_model): LoraModel(
      (model): BertForSequenceClassification(
        (bert): BertModel(
          (embeddings): BertEmbeddings(
            (word_embeddings): Embedding(119547, 768, padding_idx=0)
            (position_embeddings): Embedding(512, 768)
            (token_type_embeddings): Embedding(2, 768)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (encoder): BertEncoder(
            (layer): ModuleList(
              (0-11): 12 x BertLayer(
                (attention): BertAttention(
                  (self): BertSelfAttention(
                    (query): lora.Linear(
                      (base_layer): Linear(in_features=768, out_features=768, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.1, inplace=False)
                      )
   

INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/refs%2Fpr%2F4/model.safetensors "HTTP/1.1 302 Found"


In [5]:
# Ячейка между Model init и Forward Pass
print("=== Проверка совместимости модели и данных ===\n")

# 1. Параметры модели
total_params = sum(p.numel() for p in nlp_model.parameters())
trainable_params = sum(p.numel() for p in nlp_model.parameters() if p.requires_grad)
print(f"Всего параметров:    {total_params:,}")
print(f"Обучаемых:           {trainable_params:,}")
print(f"Заморожено:          {total_params - trainable_params:,}")
print(f"% обучаемых:         {trainable_params/total_params*100:.1f}%\n")

# 2. Что именно обучается
print("Обучаемые слои:")
for name, param in nlp_model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {list(param.shape)}")

# Ожидание для LoRA: только lora_A, lora_B слои + classifier
# Если список пустой — LoRA не применилась

=== Проверка совместимости модели и данных ===

Всего параметров:    178,151,428
Обучаемых:           296,450
Заморожено:          177,854,978
% обучаемых:         0.2%

Обучаемые слои:
  model.base_model.model.bert.encoder.layer.0.attention.self.query.lora_A.default.weight: [8, 768]
  model.base_model.model.bert.encoder.layer.0.attention.self.query.lora_B.default.weight: [768, 8]
  model.base_model.model.bert.encoder.layer.0.attention.self.value.lora_A.default.weight: [8, 768]
  model.base_model.model.bert.encoder.layer.0.attention.self.value.lora_B.default.weight: [768, 8]
  model.base_model.model.bert.encoder.layer.1.attention.self.query.lora_A.default.weight: [8, 768]
  model.base_model.model.bert.encoder.layer.1.attention.self.query.lora_B.default.weight: [768, 8]
  model.base_model.model.bert.encoder.layer.1.attention.self.value.lora_A.default.weight: [8, 768]
  model.base_model.model.bert.encoder.layer.1.attention.self.value.lora_B.default.weight: [768, 8]
  model.base_model.mod

# Forward Pass

In [6]:
sample_batch_gpu = {
    k: v.to(device) for k, v in sample_batch.items()
    if isinstance(v, torch.Tensor)
}

with torch.no_grad():
    outputs = nlp_model(**sample_batch_gpu)

# BertForSequenceClassification возвращает SequenceClassifierOutput
print("Output type:", type(outputs))
print("Has logits:", hasattr(outputs, "logits"))
print("Logits shape:", outputs.logits.shape)
# Ожидание: torch.Size([16, 2]) — batch_size x num_classes

# Проверяем что предсказания дают классы 0 и 1, а не случайные числа
preds = torch.argmax(outputs.logits, dim=-1)
print("Predictions:", preds)
print("Unique predicted classes:", preds.unique().tolist())
# Ожидание: только [0] или [1] или [0, 1]

print("Labels:", sample_batch_gpu["labels"])

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Output type: <class 'transformers.modeling_outputs.SequenceClassifierOutput'>
Has logits: True
Logits shape: torch.Size([4, 2])
Predictions: tensor([1, 0, 0, 0])
Unique predicted classes: [0, 1]
Labels: tensor([0, 0, 0, 0])


# Baseline Evaluation

In [ ]:
from sklearn.metrics import classification_report  # noqa: E402
from tqdm.auto import tqdm  # noqa: E402


all_preds = []
all_labels = []
max_batches = 50

for i, batch in enumerate(tqdm(
    val_dataloader, desc="Baseline Eval",
    total=min(len(val_dataloader), max_batches)
)):
    if i >= max_batches:
        break

    batch_gpu = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        outputs = nlp_model(**batch_gpu)

    # ИСПРАВЛЕНО: берём logits, а не pooler_output
    preds = torch.argmax(outputs.logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(batch["labels"].cpu().numpy())

print("\n--- UNTRAINED BASELINE REPORT ---")
print(classification_report(all_labels, all_preds,
      target_names=["ham (0)", "spam (1)"], zero_division=0))

Baseline Eval: 100%|██████████| 50/50 [00:18<00:00,  2.69it/s]


--- UNTRAINED BASELINE REPORT ---
              precision    recall  f1-score   support

     ham (0)       0.71      0.47      0.56       150
    spam (1)       0.21      0.42      0.28        50

    accuracy                           0.46       200
   macro avg       0.46      0.44      0.42       200
weighted avg       0.58      0.46      0.49       200



# Lora 
Все 12 слоев берта обернуты в query и value матрицы 8 ранга, классификационная голова также обучаема.(веса не заморожены)
# Classifier Head 
классификационная голова инициализирована случайно
# Baseline Metrics
Случайное угадывание с учетом дисбаланса классов, все корректно при условии необученной модели.
Отсутствует Data Leakage т.к. accuracy не завышен